# NorthWind PLUS business agent — Relate by Neo4j + Neo4j Aura + Claude (Google Colab)

Ask business questions about the **live** NorthWind PLUS simulation. The agent looks up proven Cypher templates in its
**relate** knowledge store (the ontology plus the query library), runs read queries on **Neo4j Aura**, and can make one
governed change: the L1/L2/L3 PO approval bands.

This notebook runs on Google's servers. Your six Python loops keep running wherever they are; both talk to the same
Aura database.

**Before you start**
1. Load the graph: in the Aura Query console, run `model/NorthWind_PLUS_Graph_Data_Model_Import_v2.3.2.cypher`
   from line 95 (*"// Let's clear all Nodes..."*) to line 1171. This **wipes the database** and builds NorthWind PLUS v2.3.2.
   Optionally also run the Recommendation Engine (lines 1271-1334).
   *(Already on v2.3.1 with data you want to keep? Run `model/NorthWind_PLUS_v2.3.2_Upgrade_from_v2.3.1.cypher` instead.)*
2. Run `model/NorthwindPlusGraphType_v2.3.2.cypher` in the same console, to apply the GRAPH TYPE schema.
3. Start the six Python loops (`model/NorthwindPlus_Stress_Test.py`, one process per `--loop`), so the data is moving.
4. Add these **Colab Secrets** (key icon 🔑 in the left sidebar), and switch on *Notebook access* for each:
   `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_PASSWORD`, and optionally `NEO4J_USER` (or `NEO4J_USERNAME`) and
   `NEO4J_DATABASE` (both default to `neo4j`). Names must match exactly.

Then run the cells from the top (*Runtime → Run all* works too; the upload step waits for your file).

## 1. Get the project

If the project is on GitHub, put its URL in `REPO_URL`. Otherwise leave it empty and upload `relate-agent.zip`
when the file picker appears.

In [ ]:
import os, sys, subprocess, zipfile, json, logging
from pathlib import Path

REPO_URL = ""          # e.g. "https://github.com/<your-account>/<repo>.git" once published

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK = Path("/content") if IN_COLAB else Path.cwd()

def find_project(base: Path):
    for p in [base, *base.glob("*"), *base.glob("*/*")]:
        if (p / "agent" / "agent.py").exists() and (p / "kb" / "build_kb.py").exists():
            return p
    return None

PROJECT = find_project(WORK)
if PROJECT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK / "repo")], check=True)
    PROJECT = find_project(WORK / "repo")
if PROJECT is None and IN_COLAB:
    from google.colab import files
    print("Choose relate-agent.zip ...")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(WORK / name) as z:
                z.extractall(WORK)
    PROJECT = find_project(WORK)
assert PROJECT, "Project not found: set REPO_URL or upload relate-agent.zip"
os.chdir(PROJECT)
print("Project folder:", PROJECT)

## 2. Install packages

Takes a minute the first time in a new Colab session.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
print(r.stdout[-2000:] or "Installed.")
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError("pip install failed (see above)")

## 3. Keys from Colab Secrets

Required: `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_PASSWORD`. Optional: `NEO4J_USER` (or `NEO4J_USERNAME`, as in Aura's
credentials file; default `neo4j`) and `NEO4J_DATABASE` (default `neo4j`). Each secret needs **Notebook access** switched on.

In [ ]:
REQUIRED = ["ANTHROPIC_API_KEY", "NEO4J_URI", "NEO4J_PASSWORD"]
OPTIONAL = ["NEO4J_USER", "NEO4J_USERNAME", "NEO4J_DATABASE"]

def secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception as exc:   # not defined, or Notebook access is off
            return exc
    return os.environ.get(name)

report = []
for name in REQUIRED + OPTIONAL:
    value = secret(name)
    if isinstance(value, Exception):
        report.append((name, f"not available ({type(value).__name__})"))
    elif value:
        os.environ[name] = value.strip()
        report.append((name, "loaded"))
    else:
        report.append((name, "not set"))
for name, status in report:
    print(f"  {name:<18} {status}")

if os.environ.get("NEO4J_USERNAME") and not os.environ.get("NEO4J_USER"):
    os.environ["NEO4J_USER"] = os.environ["NEO4J_USERNAME"]
missing = [n for n in REQUIRED if not os.environ.get(n)]
if missing:
    raise RuntimeError("Missing required secret(s): " + ", ".join(missing) +
                       ". In the key panel, check the exact names and switch on Notebook access for each, then rerun this cell.")
print("\nNEO4J_URI =", os.environ["NEO4J_URI"], "| user =", os.environ.get("NEO4J_USER", "neo4j"),
      "| database =", os.environ.get("NEO4J_DATABASE", "neo4j"))

## 4. Knowledge store (relate)

Built fresh in each Colab session (a few seconds).

In [ ]:
for sub in ("agent", "kb"):
    sys.path.insert(0, str(PROJECT / sub))
logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)

from IPython.display import Markdown, display
from build_kb import build
import agent as northwind_agent

store = PROJECT / "kb" / "northwind_kb.rl"
if not store.exists():
    print(json.dumps(build(store), indent=2))
else:
    print("Using existing knowledge store:", store.name)

## 5. Sign in

`EMPLOYEE_ID` stands in for a login — the session decides who you are, the model never does.

| EmployeeID | Who | Can change approval bands? |
|---|---|---|
| `25` | Olivia Bennett, ProcurementPolicyAdmin | yes |
| `12` | Gloria Gaynor, Level1Approver | no (can still ask questions) |

In [ ]:
EMPLOYEE_ID = "25"
MODEL = northwind_agent.DEFAULT_MODEL      # e.g. "claude-opus-5-5" for harder questions

session = northwind_agent.Session(EMPLOYEE_ID, MODEL)
print(f"Signed in as {session.user['name']} ({', '.join(session.user['roles']) or 'no roles'}) -- model {session.model}")


def ask(question: str):
    """Ask the agent. Tool calls print as they happen; the answer renders as Markdown."""
    display(Markdown(session.ask(question)))


def new_conversation():
    """Forget the conversation so far (same sign-in)."""
    session.messages.clear()
    print("Conversation cleared.")

## 6. Ask questions

The conversation carries over between cells, so follow-ups like *"and for last week?"* work.

In [ ]:
ask("Which suppliers get their quotes rejected most often?")

In [ ]:
ask("How many POs are waiting for approval?")

### Governed change

The agent previews first, then asks you to confirm in a text box under the cell: type `y` and press Enter to apply,
anything else to cancel. Only EmployeeID 25 can apply it.

In [ ]:
ask("Raise L1 to 8000 and L2 to 20000 to speed up vetting.")

In [ ]:
ask("Who changed the approval limits, when and why?")

### Your questions

In [ ]:
# ask("Which products are at risk of running out?")

## 7. What happened in this session

Which templates were reused, which queries were new, and the tokens each answer used (all model calls).

In [ ]:
events = [json.loads(line) for line in session.log_path.read_text(encoding="utf-8").splitlines()]
rows, totals = [], {"input_tokens": 0, "output_tokens": 0, "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}
question, tools = None, []
for e in events:
    if e["event"] == "question":
        question, tools = e["text"], []
    elif e["event"] == "tool":
        label = e["name"]
        if e["name"] == "run_read_query":
            label += f" ({e['input'].get('template') or 'new Cypher'})"
        tools.append(f"{label} {e['ms']} ms")
    elif e["event"] == "answer":
        u = e.get("usage") or {}
        for k in totals:
            totals[k] += u.get(k) or 0
        rows.append(f"| {question} | {'<br>'.join(tools) or '-'} | {u.get('model_calls', 1)} calls: {u.get('input_tokens', 0):,} in (+{u.get('cache_read_input_tokens') or 0:,} cached) / {u.get('output_tokens', 0):,} out |")
display(Markdown("| Question | Tool calls | Tokens (all model calls) |\n|---|---|---|\n" + "\n".join(rows)))
print("Session totals:", totals)

In [ ]:
# Keep the session log (Colab files disappear when the runtime ends)
if IN_COLAB:
    from google.colab import files
    files.download(str(session.log_path))

## 8. Done

In [ ]:
session.aura.close()
session.kb.close()
print("Closed the Aura connection and the knowledge store.")